
# AGN corona: bolometric luminosity sets normalization, not shape

The AGN X-ray corona produces a cut-off power-law (photon index Gamma
roughly 1.8, E_cut around 300 keV) normalized through the
alpha_OX-L_2500 relation (Lusso & Risaliti 2016). At fixed Gamma and
alpha_OX, increasing bolometric luminosity shifts the whole spectrum
upward but leaves the spectral *shape* nearly intact — the
sub-linear alpha_OX relation only steepens the shape at the top of
the quasar regime.

We sweep ``log L_bol`` from 42 to 46.5 erg/s on a single panel and
overlay the 2–10 keV band luminosity against L_bol on the right —
the canonical low-redshift correlation that lets X-ray surveys
estimate bolometric power from a single broadband flux.

Companion: ``plot_xray_nh_sweep.py`` (varies obscuration at fixed
L_bol); ``plot_alpha_ox_sweep.py`` (varies the UV-X-ray slope).

## References

- Lusso & Risaliti 2016, ApJ 819, 154 (alpha_OX-L_2500 relation).
- Just et al. 2007, ApJ 665, 1004 (X-ray bolometric corrections).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from tengri.analysis.plotting import setup_style
from tengri.xray import xray_agn_corona

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

wavelength = jnp.logspace(np.log10(0.0124), np.log10(124.0), 512)
wave_keV = 12.398 / np.array(wavelength)

log_lbol_values = np.linspace(42.0, 46.5, 10)
# Hopkins+2007 BC=5.15 at 2500 A
_BC_NU = 5.15 * 1.199e15
cmap = plt.get_cmap("viridis")
norm = mpl.colors.Normalize(vmin=log_lbol_values.min(), vmax=log_lbol_values.max())

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))

# Left panel: L_bol family of cut-off power-laws
ax = axes[0]
for log_lbol in log_lbol_values:
    L_2500 = 10.0**log_lbol / _BC_NU
    l_xray = np.asarray(xray_agn_corona(wavelength, l_2500_30deg_erg_hz=L_2500))
    ax.loglog(wave_keV, l_xray, color=cmap(norm(log_lbol)), lw=1.3)
ax.axvspan(0.5, 2.0, alpha=0.10, color="C0")
ax.axvspan(2.0, 10.0, alpha=0.10, color="C2")
ax.text(0.7, 1.5e21, "soft\n(0.5–2 keV)", fontsize=7, color="C0", ha="center")
ax.text(4.0, 1.5e21, "hard\n(2–10 keV)", fontsize=7, color="C2", ha="center")
ax.set(
    xlim=(0.1, 1000.0),
    ylim=(1.0e21, 5.0e27),
    xlabel="Energy [keV]",
    ylabel=r"$L_\nu$  [erg s$^{-1}$ Hz$^{-1}$]",
)
cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=0.01)
cbar.set_label(r"$\log_{10}(L_{\rm bol})$  [erg s$^{-1}$]")

# Right panel: integrated hard-band luminosity vs L_bol — the standard
# X-ray bolometric correction track.
ax = axes[1]
hard_mask = (wave_keV >= 2.0) & (wave_keV <= 10.0)
log_lbol_dense = np.linspace(42.0, 46.5, 60)
l_hard = []
for log_lbol in log_lbol_dense:
    L_2500 = 10.0**log_lbol / _BC_NU
    l_xray = np.asarray(xray_agn_corona(wavelength, l_2500_30deg_erg_hz=L_2500))
    # nu L_nu integrated in nu = -L_nu d(c/lam); use trapezoid on |dE|
    keV_to_erg = 1.602e-9
    nu = wave_keV[hard_mask] * keV_to_erg / 6.626e-27
    order = np.argsort(nu)
    l_hard.append(float(np.trapezoid(l_xray[hard_mask][order], nu[order])))
l_hard = np.asarray(l_hard)

ax.loglog(10.0**log_lbol_dense, l_hard, color="C2", lw=1.6)
ax.set(
    xlabel=r"$L_{\rm bol}$  [erg s$^{-1}$]",
    ylabel=r"$L_{\rm 2-10\,keV}$  [erg s$^{-1}$]",
    xlim=(1.0e42, 5.0e46),
)
ax.text(
    0.04,
    0.95,
    r"X-ray bolometric correction implied by"
    "\n"
    r"the $\alpha_{\rm OX}-L_{2500}$ relation",
    transform=ax.transAxes,
    va="top",
    fontsize=9,
)

fig.tight_layout()
plt.savefig("plot_xray_agn.png", dpi=150, bbox_inches="tight")